# Week 07 · Attention、向量与检索 API

Token 是模型处理的单位，不一定等于字或单词。Embedding 将 token/文本映射为向量，余弦相似度是两个向量夹角的余弦；先归一化后点积即可。相似度高不代表事实正确，更不代表片段能支持答案。Transformer 的 attention 用 QKᵀ 的相似度决定对 V 的加权：softmax(QKᵀ/√d)V。mask 控制能看哪些位置。

本例先计算可检查的 attention，再实现 TF-IDF + SVD 的小型潜在语义检索。它是本地教学基线，不是预训练神经 embedding，词表外和同义改写是失败点。真实模型应查看 Hugging Face model card、许可证、语言覆盖、最大长度和 pooling 方式；不能简单把任意 hidden state 当句向量。

生产模型 API 必须设置超时、有限重试和结构化输出验证。只对可安全重试的错误重试；不无限重复有副作用的生成。模型用量以服务实际返回为准。网站中的 Codex 助手使用本机配置，不要求你在 Notebook 放 API Key。

## 学习方式 / How to study
先预测代码结果，再逐行运行。改变一个输入、解释变化，最后不看参考实现重写关键函数。阅读不是掌握的证据；能独立实现、测试、解释失败才是。

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel,Field

rng=np.random.default_rng(42)
Q,K,V=[rng.normal(size=(3,4)) for _ in range(3)]
scores=Q@K.T/np.sqrt(4)
scores-=scores.max(axis=1,keepdims=True)  # 防止 exp 溢出
weights=np.exp(scores);weights/=weights.sum(axis=1,keepdims=True)
assert np.allclose(weights.sum(axis=1),1)
print("注意力权重：\n",weights,"\n加权表示：\n",weights@V)

documents=["Python virtual environments isolate project dependencies.",
 "SQL transactions commit or roll back database changes.",
 "FastAPI validates HTTP requests and JSON responses.",
 "Neural networks learn weights using gradients and optimizers.",
 "Vector search compares normalized text embeddings.",
 "Git branches isolate source code changes for review."]
def chunks(text,size=9,overlap=2):
    if not 0<=overlap<size:raise ValueError("invalid overlap")
    words=text.split()
    return [" ".join(words[i:i+size]) for i in range(0,len(words),size-overlap)]
index=[{"document":i,"paragraph":j,"text":chunk} for i,d in enumerate(documents) for j,chunk in enumerate(chunks(d))]
vectorizer=TfidfVectorizer();matrix=vectorizer.fit_transform([x["text"] for x in index])
svd=TruncatedSVD(n_components=4,random_state=42)
vectors=normalize(svd.fit_transform(matrix))
def search(query,k=3):
    encoded=vectorizer.transform([query])
    if encoded.nnz==0:return []  # 不为完全未知的查询假造相关性
    query_vector=normalize(svd.transform(encoded))
    similarities=(vectors@query_vector.T).ravel()
    return [{**index[i],"score":float(similarities[i])} for i in np.argsort(-similarities)[:k]]
print(search("database transactions"))
app=FastAPI()
class Query(BaseModel):
    text:str=Field(min_length=1,max_length=500)
    k:int=Field(default=3,ge=1,le=10)
@app.post("/search")
def route(body:Query):return search(body.text,body.k)
with TestClient(app) as client:
    assert client.post("/search",json={"text":"SQL database"}).status_code==200
    assert client.post("/search",json={"text":"SQL","k":0}).status_code==422
assert search("zzunknownzz")==[]

## 练习 / Exercises
记录 10 个查询的预期文档与实际首位结果。替换 SVD 为经过许可审核的句向量模型时，哪些索引需要重建？

先在下面独立完成，再展开参考实现。

In [ ]:
# 在这里写你的实现；运行后检查边界。


## 参考实现与验收 / Reference and checks
参考实现是一个可行方案，不是唯一答案。不要在未完成练习前直接复制。

In [ ]:
queries={"SQL database":1,"Python dependencies":0,"HTTP JSON":2,"gradients optimizers":3,"embeddings":4,"Git branches":5}
hits=[]
for query,expected in queries.items():
    result=search(query,1)
    hit=bool(result and result[0]["document"]==expected)
    hits.append(hit);print(query,hit,result)
print("当前小样本 Recall@1：",sum(hits)/len(hits),"不是生产评测结论")